<a href="https://colab.research.google.com/github/praneetha-glitch/AI-Lab/blob/main/ForwardChaining_week_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

def parse(atom):
    name, values = atom.strip().split("(")
    args = values[:-1].split(",")
    return name.strip(), [a.strip() for a in args]


# Unification function
def unify(pattern, fact, sub):
    p, pa = parse(pattern)
    f, fa = parse(fact)

    if p != f or len(pa) != len(fa):
        return None

    sub = sub.copy()

    for a, b in zip(pa, fa):
        if a[0].islower():  # variable
            if a in sub:
                if sub[a] != b:
                    return None
            else:
                sub[a] = b
        elif a != b:
            return None

    return sub


# Replace variables with values
def substitute(atom, sub):
    name, args = parse(atom)
    args = [sub.get(a, a) for a in args]
    return name + "(" + ",".join(args) + ")"


# Match all conditions of a rule
def match(conditions, facts, sub=None):
    if sub is None:
        sub = {}

    if not conditions:
        return [sub]

    results = []

    first = substitute(conditions[0], sub)

    for fact in facts:
        new_sub = unify(first, fact, sub)

        if new_sub is not None:
            results.extend(match(conditions[1:], facts, new_sub))

    return results


# Forward Chaining
def forward_chaining(facts, rules, query):

    while True:
        new = []

        for rule in rules:
            parts = rule.split("->")
            conditions = parts[0].split("AND")
            conclusion = parts[1].strip()

            substitutions = match(conditions, facts)

            for sub in substitutions:
                result = substitute(conclusion, sub)

                if result not in facts and result not in new:
                    new.append(result)

        if not new:
            return query in facts

        facts.extend(new)

        if query in facts:
            return True


# User input
n = int(input("Enter number of facts: "))
facts = []

for i in range(n):
    facts.append(input("Enter fact: ").strip())

m = int(input("Enter number of rules: "))
rules = []

for i in range(m):
    rules.append(input("Enter rule: ").strip())

query = input("Enter query: ").strip()

if forward_chaining(facts, rules, query):
    print("Query is TRUE")
else:
    print("Query is FALSE")


Enter number of facts: 5
Enter fact: American(Robert)
Enter fact: Enemy(A, America)
Enter fact: Missile(M1)
Enter fact: Owns(A, M1)
Enter fact: Sells(Robert, M1, A)
Enter number of rules: 3
Enter rule:  Enemy(x, America) -> Hostile(x)
Enter rule: Missile(x) -> Weapon(x)
Enter rule: American(x) AND Weapon(y) AND Sells(x, y, z) AND Hostile(z) -> Criminal(x)
Enter query: Criminal(Robert)
Query is TRUE
